In [1]:
import pandas as pd


ohlcv_canonical = pd.read_parquet("/Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/data/raw_canonical/ohlcv.parquet")
ohlcv_canonical

,date,ticker,open,high,low,close,volume,value,frequency,foreign_buy,foreign_sell
0,2015-01-02,AALI,16715.333882,17196.845842,16680.940171,16904.500000,1385945.0,NaN,NaN,NaN,NaN
1,2015-01-02,ABBA,48.466820,50.055897,48.466820,49.261356,117049.0,NaN,NaN,NaN,NaN
2,2015-01-02,ABDA,5533.175293,5533.175293,5533.175293,5533.175293,5000.0,NaN,NaN,NaN,NaN
3,2015-01-02,ABMM,2110.177002,2110.177002,2110.177002,2110.177002,0.0,NaN,NaN,NaN,NaN
4,2015-01-02,ACES,576.700535,580.373786,569.354031,573.027283,15328600.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1956368,2026-06-05,ZATA,59.000000,60.000000,51.000000,51.000000,130311000.0,7.050457e+09,6689.0,15319800.0,16314000.0
1956369,2026-06-05,ZBRA,0.000000,0.000000,0.000000,50.000000,0.0,0.000000e+00,0.0,0.0,0.0
1956370,2026-06-05,ZINC,0.000000,0.000000,0.000000,30.000000,0.0,0.000000e+00,0.0,0.0,0.0
1956371,2026-06-05,ZONE,0.000000,348.000000,340.000000,340.000000,49600.0,1.713340e+07,31.0,0.0,100.0


├── 2020
│   ├── Agu
│   ├── Apr
...........
│   ├── Okt
│   └── Sep

...........

└── 2026
    ├── Feb
    ├── Jan
    └── Mar

In [13]:
import pandas as pd

base_score_18_may = pd.read_csv('signals/daily/signal_18_may_2026/continual_model/all_scores.csv')
continual_model_18_may = pd.read_csv('signals/daily/signal_18_may_2026/continual_model/all_strategy_watchlist.csv')

base_score_19_may = pd.read_csv('signals/daily/signal_19_may_2026/continual_model/all_scores.csv')
continual_model_19_may = pd.read_csv('signals/daily/signal_19_may_2026/continual_model/all_strategy_watchlist.csv')

base_score_20_may = pd.read_csv('signals/daily/signal_20_may_2026/continual_model/all_scores.csv')
continual_model_20_may = pd.read_csv('signals/daily/signal_20_may_2026/continual_model/all_strategy_watchlist.csv')

base_score_21_may = pd.read_csv('signals/daily/signal_21_may_2026/continual_model/all_scores.csv')
continual_model_21_may = pd.read_csv('signals/daily/signal_21_may_2026/continual_model/all_strategy_watchlist.csv')

base_score_22_may = pd.read_csv('signals/daily/signal_22_may_2026/continual_model/all_scores.csv')
continual_model_22_may = pd.read_csv('signals/daily/signal_22_may_2026/continual_model/all_strategy_watchlist.csv')

base_score_25_may = pd.read_csv('signals/daily/signal_25_may_2026/continual_model/all_scores.csv')
continual_model_25_may = pd.read_csv('signals/daily/signal_25_may_2026/continual_model/all_strategy_watchlist.csv')


base_score_26_may = pd.read_csv('signals/daily/signal_26_may_2026/continual_model/all_scores.csv')
continual_model_26_may = pd.read_csv('signals/daily/signal_26_may_2026/continual_model/all_strategy_watchlist.csv')


In [5]:
import pandas as pd
from pathlib import Path

def filter_scores_by_watchlist(
    base_score_df,
    watchlist_df,
    output_path=None,
    ticker_col='ticker'
):
    """
    Filter base_score berdasarkan ticker unik
    dari watchlist dataframe.

    Parameters
    ----------
    base_score_df : pd.DataFrame
        DataFrame all_scores.csv

    watchlist_df : pd.DataFrame
        DataFrame all_strategy_watchlist.csv

    output_path : str or None
        Path output csv.
        Jika None -> tidak save file.

    ticker_col : str
        Nama kolom ticker

    Returns
    -------
    filtered_df : pd.DataFrame
        Hasil dataframe yang sudah difilter

    ticker_list : list
        List ticker unik
    """

    # Ambil unique ticker
    ticker_list = (watchlist_df[ticker_col] .dropna() .astype(str) .str.strip() .unique() .tolist())

    print(f'Jumlah ticker unik: {len(ticker_list)}')
    print(ticker_list)

    # Filter dataframe
    filtered_df = base_score_df[
        base_score_df[ticker_col].isin(ticker_list)
    ]

    print(f'Shape original : {base_score_df.shape}')
    print(f'Shape filtered : {filtered_df.shape}')

    # Save jika output_path diberikan
    if output_path is not None:

        output_path = Path(output_path)

        # buat folder jika belum ada
        output_path.parent.mkdir(parents=True, exist_ok=True)

        filtered_df.to_csv(output_path, index=False)

        print(f'Filtered CSV saved to: {output_path}')

    return filtered_df, ticker_list

In [15]:
import os
import shutil
from pathlib import Path

def copy_all_strategy_watchlists(
    source_root='signals/daily',
    destination_dir='/Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass'
):
    """
    Copy semua file all_strategy_watchlist.csv
    dari seluruh folder continual_model.

    Contoh:
    signals/daily/signal_25_may_2026/continual_model/all_strategy_watchlist.csv

    akan dicopy menjadi:
    signals/collected_watchlists/signal_25_may_2026_all_strategy_watchlist.csv
    """

    source_root = Path(source_root)
    destination_dir = Path(destination_dir)

    # buat folder tujuan kalau belum ada
    destination_dir.mkdir(parents=True, exist_ok=True)

    copied_files = []

    # cari semua continual_model
    for continual_dir in source_root.glob('signal_*/continual_model'):

        source_file = continual_dir / 'all_strategy_watchlist.csv'

        # cek apakah file ada
        if source_file.exists():

            # ambil nama signal folder
            signal_name = continual_dir.parent.name

            # rename file agar tidak overwrite
            destination_file = (
                destination_dir /
                f'{signal_name}_all_strategy_watchlist.csv'
            )

            # copy file
            shutil.copy2(source_file, destination_file)

            copied_files.append(str(destination_file))

            print(f'Copied: {source_file}')
            print(f'   --> {destination_file}')

    print(f'\nTotal copied files: {len(copied_files)}')

    return copied_files


def copy_all_numeric_trade_plan(
    source_root='signals/daily',
    destination_dir='/Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass'
):
    """
    Copy semua file numeric_trade_plan.csv
    dari seluruh folder continual_model.

    Contoh:
    signals/daily/signal_25_may_2026/continual_model/numeric_trade_plan.csv

    akan dicopy menjadi:
    signals/collected_watchlists/signal_25_may_2026_numeric_trade_plan.csv
    """

    source_root = Path(source_root)
    destination_dir = Path(destination_dir)

    # buat folder tujuan kalau belum ada
    destination_dir.mkdir(parents=True, exist_ok=True)

    copied_files = []

    # cari semua continual_model
    for continual_dir in source_root.glob('signal_*/continual_model'):

        source_file = continual_dir / 'numeric_trade_plan.json'

        # cek apakah file ada
        if source_file.exists():

            # ambil nama signal folder
            signal_name = continual_dir.parent.name

            # rename file agar tidak overwrite
            destination_file = (
                destination_dir /
                f'{signal_name}_numeric_trade_plan.json'
            )

            # copy file
            shutil.copy2(source_file, destination_file)

            copied_files.append(str(destination_file))

            print(f'Copied: {source_file}')
            print(f'   --> {destination_file}')

    print(f'\nTotal copied files: {len(copied_files)}')

    return copied_files


def copy_all_base_score(
    source_root='signals/daily',
    destination_dir='/Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass'
):
    """
    Copy semua file all_scores.csv
    dari seluruh folder continual_model.

    Contoh:
    signals/daily/signal_25_may_2026/continual_model/all_scores.csv

    akan dicopy menjadi:
    signals/collected_watchlists/signal_25_may_2026_all_scores.csv
    """

    source_root = Path(source_root)
    destination_dir = Path(destination_dir)

    # buat folder tujuan kalau belum ada
    destination_dir.mkdir(parents=True, exist_ok=True)

    copied_files = []

    # cari semua continual_model
    for continual_dir in source_root.glob('signal_*/continual_model'):

        source_file = continual_dir / 'all_scores.csv'

        # cek apakah file ada
        if source_file.exists():

            # ambil nama signal folder
            signal_name = continual_dir.parent.name

            # rename file agar tidak overwrite
            destination_file = (
                destination_dir /
                f'{signal_name}_all_scores.csv'
            )

            # copy file
            shutil.copy2(source_file, destination_file)

            copied_files.append(str(destination_file))

            print(f'Copied: {source_file}')
            print(f'   --> {destination_file}')

    print(f'\nTotal copied files: {len(copied_files)}')

    return copied_files

In [ ]:
filtered_df, ticker_list = filter_scores_by_watchlist(
    base_score_df=base_score_26_may,
    watchlist_df=continual_model_26_may,
    output_path='bitchass/filtered_scores_26_may.csv'
)

Jumlah ticker unik: 16
['MSJA', 'CUAN', 'BEEF', 'MSIN', 'TIRA', 'CYBR', 'GZCO', 'TRUE', 'BIPI', 'MBMA', 'BELL', 'HBAT', 'MDIA', 'BNBR', 'KOKA', 'ASPR']
Shape original : (229, 205)
Shape filtered : (16, 205)
Filtered CSV saved to: bitchass/filtered_scores_18_may.csv


In [14]:
copied_files = copy_all_strategy_watchlists()

Copied: signals/daily/signal_25_may_2026/continual_model/all_strategy_watchlist.csv
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_25_may_2026_all_strategy_watchlist.csv
Copied: signals/daily/signal_22_may_2026/continual_model/all_strategy_watchlist.csv
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_22_may_2026_all_strategy_watchlist.csv
Copied: signals/daily/signal_26_may_2026/continual_model/all_strategy_watchlist.csv
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_26_may_2026_all_strategy_watchlist.csv
Copied: signals/daily/signal_18_may_2026/continual_model/all_strategy_watchlist.csv
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_18_may_2026_all_strategy_watchlist.csv
Copied: signals/daily/signal_19_may_2026/continual_model/all_strategy_watchlist.csv
   --> /Users/albert/Documents/F

In [16]:
copied_numeric_trade_plan_files = copy_all_numeric_trade_plan()

Copied: signals/daily/signal_25_may_2026/continual_model/numeric_trade_plan.json
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_25_may_2026_numeric_trade_plan.json
Copied: signals/daily/signal_22_may_2026/continual_model/numeric_trade_plan.json
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_22_may_2026_numeric_trade_plan.json
Copied: signals/daily/signal_18_may_2026/continual_model/numeric_trade_plan.json
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_18_may_2026_numeric_trade_plan.json
Copied: signals/daily/signal_19_may_2026/continual_model/numeric_trade_plan.json
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_19_may_2026_numeric_trade_plan.json
Copied: signals/daily/signal_21_may_2026/continual_model/numeric_trade_plan.json
   --> /Users/albert/Documents/Finances/projects/02_alpha_r

In [11]:
copied_all_scores_files = copy_all_base_score()

Copied: signals/daily/signal_25_may_2026/continual_model/all_scores.csv
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_25_may_2026_all_scores.csv
Copied: signals/daily/signal_22_may_2026/continual_model/all_scores.csv
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_22_may_2026_all_scores.csv
Copied: signals/daily/signal_26_may_2026/continual_model/all_scores.csv
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_26_may_2026_all_scores.csv
Copied: signals/daily/signal_18_may_2026/continual_model/all_scores.csv
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_18_may_2026_all_scores.csv
Copied: signals/daily/signal_19_may_2026/continual_model/all_scores.csv
   --> /Users/albert/Documents/Finances/projects/02_alpha_research/alpha_research/bitchass/signal_19_may_2026_all_scores.csv
Copied: signals

In [2]:
import pandas as pd

macro = pd.read_parquet("data/raw_canonical/macro.parquet")

macro

,date,idr_usd,usd_idr,wti,brent,coal_proxy,ihsg,bi_rate,macro_missing_flag,coal_gap_flag,oil_avg,wti_return,brent_return,fx_return,coal_proxy_return,bi_rate_change
0,2015-01-01,0.000080,12500.000000,52.689999,56.419998,119.758713,5242.564941,7.75,0,0,54.554998,NaN,NaN,NaN,NaN,NaN
1,2015-01-02,0.000080,12500.000000,52.689999,56.419998,145.800003,5242.564941,7.75,0,0,54.554998,0.000000,0.000000,0.000000,0.217448,0.0
2,2015-01-03,0.000080,12500.000000,52.689999,56.419998,119.758713,5242.564941,7.75,0,0,54.554998,0.000000,0.000000,0.000000,-0.178610,0.0
3,2015-01-04,0.000080,12500.000000,52.689999,56.419998,119.758713,5242.564941,7.75,0,0,54.554998,0.000000,0.000000,0.000000,0.000000,0.0
4,2015-01-05,0.000079,12658.227848,50.040001,53.110001,143.500000,5219.791992,7.75,0,0,51.575001,-0.050294,-0.058667,-0.012500,0.198243,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4158,2026-05-21,0.000057,17543.859649,96.349998,102.580002,216.531893,6094.940918,4.75,0,0,99.465000,-0.019438,-0.023234,0.017857,-0.023079,0.0
4159,2026-05-22,0.000057,17543.859649,96.599998,103.540001,218.544505,6162.044922,4.75,0,0,100.070000,0.002595,0.009359,0.000000,0.009295,0.0
4160,2026-05-23,0.000057,17543.859649,96.599998,103.540001,218.544505,6162.044922,4.75,0,0,100.070000,0.000000,0.000000,0.000000,0.000000,0.0
4161,2026-05-24,0.000057,17543.859649,96.599998,103.540001,218.544505,6162.044922,4.75,0,0,100.070000,0.000000,0.000000,0.000000,0.000000,0.0


In [1]:
from pathlib import Path
import pandas as pd

p = Path("data/pure_raw/trading_summary/RingkasanSaham-20260602.xlsx")

print("exists:", p.exists())
print("size:", p.stat().st_size if p.exists() else None)

xl = pd.ExcelFile(p)
print("sheets:", xl.sheet_names)

for sheet in xl.sheet_names:
    print("\n=== SHEET:", sheet, "===")
    try:
        df = pd.read_excel(p, sheet_name=sheet, nrows=10)
        print("shape sample:", df.shape)
        print("columns:", list(df.columns))
        print(df.head(3))
    except Exception as e:
        print("ERROR:", e)

exists: True
size: 174520
sheets: ['Sheet1']

=== SHEET: Sheet1 ===
shape sample: (10, 32)
columns: ['No', 'IDStockSummary', 'Date', 'StockCode', 'StockName', 'Remarks', 'Previous', 'OpenPrice', 'FirstTrade', 'High', 'Low', 'Close', 'Change', 'Volume', 'Value', 'Frequency', 'IndexIndividual', 'Offer', 'OfferVolume', 'Bid', 'BidVolume', 'ListedShares', 'TradebleShares', 'WeightForIndex', 'ForeignSell', 'ForeignBuy', 'DelistingDate', 'NonRegularVolume', 'NonRegularValue', 'NonRegularFrequency', 'persen', 'percentage']
   No  IDStockSummary                 Date StockCode  \
0   1         4023042  2026-06-02T00:00:00      AADI   
1   2         4023043  2026-06-02T00:00:00      AALI   
2   3         4023044  2026-06-02T00:00:00      ABBA   

                      StockName                         Remarks  Previous  \
0  Adaro Andalan Indonesia Tbk.  CDMO1SD0F10000A121------------      8400   
1       Astra Agro Lestari Tbk.  --MO113E100000D232------------      6475   
2             Mahaka M

In [2]:
from pathlib import Path
import pandas as pd

files = [
    "data/pure_raw/trading_summary/RingkasanSaham-20260529.xlsx",
    "data/pure_raw/trading_summary/RingkasanSaham-20260602.xlsx",
]

for fp in files:
    p = Path(fp)
    print("\n\nFILE:", p)
    print("exists:", p.exists())
    print("size:", p.stat().st_size if p.exists() else None)
    
    xl = pd.ExcelFile(p)
    print("sheets:", xl.sheet_names)
    
    for sheet in xl.sheet_names[:3]:
        try:
            df = pd.read_excel(p, sheet_name=sheet, nrows=5)
            print("\nSHEET:", sheet)
            print("columns:", list(df.columns))
            print(df.head(2))
        except Exception as e:
            print("ERROR reading sheet:", sheet, e)



FILE: data/pure_raw/trading_summary/RingkasanSaham-20260529.xlsx
exists: True
size: 174725
sheets: ['Sheet1']

SHEET: Sheet1
columns: ['No', 'IDStockSummary', 'Date', 'StockCode', 'StockName', 'Remarks', 'Previous', 'OpenPrice', 'FirstTrade', 'High', 'Low', 'Close', 'Change', 'Volume', 'Value', 'Frequency', 'IndexIndividual', 'Offer', 'OfferVolume', 'Bid', 'BidVolume', 'ListedShares', 'TradebleShares', 'WeightForIndex', 'ForeignSell', 'ForeignBuy', 'DelistingDate', 'NonRegularVolume', 'NonRegularValue', 'NonRegularFrequency', 'persen', 'percentage']
   No  IDStockSummary                 Date StockCode  \
0   1         4022083  2026-05-29T00:00:00      AADI   
1   2         4022084  2026-05-29T00:00:00      AALI   

                      StockName                         Remarks  Previous  \
0  Adaro Andalan Indonesia Tbk.  --MO1SD0F10000A121------------      8325   
1       Astra Agro Lestari Tbk.  --MO113E100000D232------------      6575   

   OpenPrice  FirstTrade  High  ...  Trad